In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "source": ["# 02 - Transform\nClean and conform Bronze → Silver data."]
  },
  {
   "cell_type": "code",
   "source": [
    "from pyspark.sql.functions import col, to_date, when\n",
    "\n",
    "catalog = \"main\"\n",
    "bronze_schema = \"bronze\"\n",
    "silver_schema = \"silver\"\n",
    "\n",
    "bronze_table = f\"{catalog}.{bronze_schema}.events_bronze\"\n",
    "silver_table = f\"{catalog}.{silver_schema}.events_silver\"\n",
    "\n",
    "df = spark.table(bronze_table)\n",
    "\n",
    "df_clean = (\n",
    "    df.dropDuplicates()\n",
    "      .filter(col(\"eventId\").isNotNull())\n",
    "      .filter(col(\"timestamp\").isNotNull())\n",
    ")\n",
    "\n",
    "df_transformed = (\n",
    "    df_clean.withColumn(\"event_date\", to_date(col(\"timestamp\")))\n",
    "            .withColumn(\"event_type\",\n",
    "                when(col(\"type\").isNull(), \"unknown\").otherwise(col(\"type\"))\n",
    "            )\n",
    ")\n",
    "\n",
    "df_transformed.write.format(\"delta\").mode(\"overwrite\").option(\"overwriteSchema\", \"true\").saveAsTable(silver_table)\n",
    "\n",
    "display(df_transformed)"
   ]
  }
 ],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 5
}